Lab | Model Conversions & Inferencing


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort
import time

device = "cpu"  # we'll deliberately use CPU to demonstrate quantisation gains
torch.manual_seed(42)

In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
model.load_state_dict(torch.load("best_fine_tuned.pth", map_location=device))
model.eval()

In [ ]:
# ── Task 1 · Part A ──────────────────────────────────────────
import os

# 1. Example input
example = torch.randn(1, 3, 224, 224)

# 2. Export
torch.onnx.export(
    model, example, "flowers_resnet18.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
print("Export OK.")

# 3. Validate
onnx_model = onnx.load("flowers_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

# 4. File size
size_mb = os.path.getsize("flowers_resnet18.onnx") / (1024 * 1024)
print(f"File size: {size_mb:.2f} MB")


# ── Task 1 · Part B ──────────────────────────────────────────

# 1. ONNX Runtime session
session = ort.InferenceSession("flowers_resnet18.onnx")

# 2. Comparison across 8 random images
model.eval()
max_diff = 0.0

for _ in range(8):
    x = torch.randn(1, 3, 224, 224)
    
    # PyTorch output
    with torch.no_grad():
        pt_out = model(x).numpy()
    
    # ONNX output
    onnx_out = session.run(None, {"input": x.numpy()})[0]
    
    diff = np.abs(pt_out - onnx_out).max()
    max_diff = max(max_diff, diff)

print(f"Max absolute difference: {max_diff:.2e}")

# 3. Assert
assert max_diff < 1e-4, f"Difference is too large: {max_diff}"
print("✅ Numerical equivalence verified!")

In [ ]:
import importlib
import inference
importlib.reload(inference)
from inference import FlowerClassifier

classifier = FlowerClassifier("flowers_resnet18.onnx")
import importlib, sys
from torchvision import transforms

# Import from inference.py
sys.path.insert(0, ".")
from inference import FlowerClassifier

classifier = FlowerClassifier("flowers_resnet18.onnx")

# Take 5 images from the validation set
from torchvision.datasets import Flowers102
val_dataset = Flowers102(root="./data", split="val", download=True)
indices = [0, 10, 50, 100, 200]

# PyTorch preprocessing (for comparison)
pt_transform = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

print("=" * 50)
for idx in indices:
    img_pil, label = val_dataset[idx]
    
    # Save temporary image (FlowerClassifier expects a file path)
    tmp_path = f"tmp_img_{idx}.png"
    img_pil.save(tmp_path)
    
    # ── ONNX prediction
    onnx_preds = classifier.predict(tmp_path, k=3)
    
    # ── PyTorch prediction
    x_pt = pt_transform(img_pil).unsqueeze(0)
    with torch.no_grad():
        pt_logits = model(x_pt).numpy()[0]
    pt_probs = np.exp(pt_logits - pt_logits.max())
    pt_probs /= pt_probs.sum()
    pt_top3 = np.argsort(pt_probs)[::-1][:3]
    
    print(f"\nImage #{idx}  |  True Label: {label}")
    print(f"  ONNX    top-3: {[(c, f'{p:.3f}') for c, p in onnx_preds]}")
    print(f"  PyTorch top-3: {[(int(c), f'{pt_probs[c]:.3f}') for c in pt_top3]}")
    
    # Verify that they are the same
    assert [c for c, _ in onnx_preds] == [int(c) for c in pt_top3], \
        f"Predictions for Image #{idx} do not match!"

print("\n" + "=" * 50)
print("✅ ONNX and PyTorch predictions match for all 5 images!")

In [ ]:
# ── Task 3 · Part 1 — Quantization ─────────────────────────
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)

fp32_size = os.path.getsize("flowers_resnet18.onnx") / (1024 * 1024)
int8_size = os.path.getsize("flowers_resnet18.int8.onnx") / (1024 * 1024)
print(f"FP32 size : {fp32_size:.2f} MB")
print(f"INT8 size : {int8_size:.2f} MB")
print(f"Size ratio: Reduced by {fp32_size / int8_size:.2f}x")


# ── Task 3 · Part 2 — Quality Comparison ─────────────────
session_fp32 = ort.InferenceSession("flowers_resnet18.onnx")
session_int8 = ort.InferenceSession("flowers_resnet18.int8.onnx")

from torchvision.datasets import Flowers102
from torchvision import transforms

val_dataset = Flowers102(root="./data", split="val", download=False)
pt_transform = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

max_diff = 0.0
mean_diffs = []
fp32_correct = 0
int8_correct = 0
total = 0

for idx in range(len(val_dataset)):
    img_pil, label = val_dataset[idx]
    x = pt_transform(img_pil).unsqueeze(0).numpy()

    out_fp32 = session_fp32.run(None, {"input": x})[0][0]
    out_int8 = session_int8.run(None, {"input": x})[0][0]

    diff = np.abs(out_fp32 - out_int8)
    max_diff = max(max_diff, diff.max())
    mean_diffs.append(diff.mean())

    fp32_correct += int(np.argmax(out_fp32) == label)
    int8_correct += int(np.argmax(out_int8) == label)
    total += 1

print(f"\nMax   absolute difference : {max_diff:.4f}")
print(f"Mean absolute difference : {np.mean(mean_diffs):.4f}")
print(f"\nFP32 accuracy: {fp32_correct/total*100:.2f}%")
print(f"INT8 accuracy: {int8_correct/total*100:.2f}%")
print(f"Accuracy difference: {abs(fp32_correct-int8_correct)/total*100:.2f}%")


# ── Task 3 · Part 3 — Latency Benchmark ────────────────────
import time

# Test image
img_pil, _ = val_dataset[0]
x_np = pt_transform(img_pil).unsqueeze(0).numpy()
x_pt = pt_transform(img_pil).unsqueeze(0)

N = 100

# PyTorch
model.eval()
with torch.no_grad():
    for _ in range(10):  # warmup
        _ = model(x_pt)
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N):
        _ = model(x_pt)
pt_ms = (time.perf_counter() - t0) / N * 1000

# ONNX FP32
for _ in range(10):  # warmup
    session_fp32.run(None, {"input": x_np})
t0 = time.perf_counter()
for _ in range(N):
    session_fp32.run(None, {"input": x_np})
fp32_ms = (time.perf_counter() - t0) / N * 1000

# ONNX INT8
for _ in range(10):  # warmup
    session_int8.run(None, {"input": x_np})
t0 = time.perf_counter()
for _ in range(N):
    session_int8.run(None, {"input": x_np})
int8_ms = (time.perf_counter() - t0) / N * 1000

print("\n| Model          | File (MB) | Latency (ms) | Speedup |")
print("|----------------|-----------|--------------|---------|")
print(f"| PyTorch FP32   | {fp32_size:.2f}     | {pt_ms:.2f}          | 1.00x   |")
print(f"| ONNX FP32      | {fp32_size:.2f}     | {fp32_ms:.2f}          | {pt_ms/fp32_ms:.2f}x   |")
print(f"| ONNX INT8      | {int8_size:.2f}     | {int8_ms:.2f}          | {pt_ms/int8_ms:.2f}x   |")

Accuracy vs Size: The INT8 model is 4x smaller than FP32 (42.83 MB → 10.76 MB), yet accuracy dropped by only 0.20% (87.35% → 87.16%). This is an excellent trade-off for deployment scenarios where storage and memory are constrained.

Latency: ONNX FP32 is 2x faster than PyTorch, which is expected due to graph-level optimizations in ONNX Runtime. However, ONNX INT8 was surprisingly 8x slower than PyTorch. This is because the CPU lacks specialized INT8 instructions (such as AVX-512 VNNI), forcing the runtime to emulate 8-bit operations in software. In practice, INT8 speedups are most significant on ARM processors (mobile devices) or dedicated AI accelerators, not on general-purpose x86 CPUs.